# Simulator_Visual.ipynb -- Validazione operativa CF_FSNN pre-FPGA

**Branch git**: `Visualizer_Building` (creato da `main` consolidato post-merge F2/Arch).

**Contesto**: dopo la chiusura di F2 EventProp (vedi `document/EVENTPROP_OPTIMIZER_SWEEP.md`) abbiamo un floor val_data ~0.222 m/s² RMSE su accel, **architetturale e non riducibile**. Questo notebook traduce quella metric astratta in **validazione operativa**: il veicolo "guida bene"? L'errore e' sistematico? In quali scenari fallisce?

**5 viste sul sistema**:
1. **Layout A statico** (5 panel + metric overlay) -- per paper figures e debug numerico
2. **Layout B top-down snapshot** -- vista spaziale a singolo frame
3. **Layout C animazione** -- replay con GIF/MP4 export per slide deck
4. **Multi-scenario aggregate** -- 20+ scenari per tipo, statistiche operative
5. **Interactive ipywidgets** -- dropdown scenario + slider tempo + play

**Metric chiave** (vedi `utils/simulator/metrics.py`):
- Spaziali: gap_rmse, gap_max_err, pos_cum_err
- Comfort: jerk_max, jerk_p95
- Safety: TTC_min
- ML reference: accel_rmse_masked (= val_data semantics)
- Per-param IDM: rmse(v0/T/s0/a/b)
- Activity: spike_rate_avg

**Checkpoint usato**: `SW_baseline_adamw_lr5e-3` (val_data 0.2218) o fallback `GRID2x2_baseline` (0.2233).
**Cache**: `data/cache_1500_highway_cut0.0_ou0.0.pt` (300 val highway scenarios).

**Tempi notebook completo** (Azure CPU): ~10-15 min totali (forward + plot + GIF export per ~5 scenari).

In [ ]:
# ===========================================================
# CELLA 0 -- Bootstrap deps + git checkout Visualizer_Building
# ===========================================================
import sys

print('Installing/upgrading dependencies (idempotent)...')
!{sys.executable} -m pip install --quiet matplotlib pandas pillow ipywidgets
!{sys.executable} -m pip install --quiet nbstripout
!nbstripout --install --attributes .gitattributes 2>/dev/null
print('   [OK] deps installed + nbstripout attivato')

print('\ngit fetch + checkout Visualizer_Building + pull:')
!git fetch origin
!git checkout Visualizer_Building 2>/dev/null || git checkout -b Visualizer_Building origin/Visualizer_Building
!git pull --no-rebase --no-edit origin Visualizer_Building

print('\nBranch + commit attuale:')
!git branch --show-current
!git log --oneline -3

In [ ]:
# ===========================================================
# CELLA 1 -- ENV CHECK + verifica utils/simulator + checkpoint
# ===========================================================
import sys, os, platform

print(f'Python:          {sys.version.split()[0]} ({platform.system()})')
print(f'Working dir:     {os.getcwd()}')

checks = []
for mod in ['torch', 'numpy', 'pandas', 'matplotlib', 'PIL']:
    try:
        m = __import__(mod)
        v = getattr(m, '__version__', '?')
        print(f'   [OK] {mod:<14} v{v}')
        checks.append((mod, True))
    except Exception as e:
        print(f'   [FAIL] {mod:<14} {e}')
        checks.append((mod, False))

# ipywidgets check (per Cella 7 opzionale)
try:
    import ipywidgets
    print(f'   [OK] ipywidgets  v{ipywidgets.__version__} (per interactive widget)')
except ImportError:
    print(f'   [WARN] ipywidgets non disponibile (Cella 7 verra' saltata)')

import torch
print(f'\nCUDA: ' + ('[OK] ' + torch.cuda.get_device_name(0) if torch.cuda.is_available() else '[INFO] CPU only'))

print('\nFile critici:')
for f in ['core/network.py', 'core/eventprop.py', 'utils/simulator/__init__.py',
          'utils/simulator/engine.py', 'utils/simulator/metrics.py',
          'utils/simulator/plots.py', 'utils/simulator/anim.py']:
    exists = os.path.isfile(f)
    print(f'   {"[OK]" if exists else "[MISSING]"} {f}')
    checks.append((f, exists))

# Cache F2
CACHE_PATH = 'data/cache_1500_highway_cut0.0_ou0.0.pt'
cache_ok = os.path.isfile(CACHE_PATH)
print(f'   {"[OK]" if cache_ok else "[MISSING]"} cache: {CACHE_PATH}')
checks.append(('cache F2', cache_ok))

# Checkpoint detection: cerca prima SW_baseline_adamw_lr5e-3, poi fallback GRID2x2_baseline
CKPT_CANDIDATES = [
    'checkpoints/SW_baseline_adamw_lr5e-3/best_model.pt',
    'checkpoints/GRID2x2_baseline/best_model.pt',
]
CKPT_PATH = None
for c in CKPT_CANDIDATES:
    if os.path.isfile(c):
        CKPT_PATH = c
        break
if CKPT_PATH is None:
    print(f'\n[ERROR] Nessun checkpoint trovato in:')
    for c in CKPT_CANDIDATES:
        print(f'   {c}')
    print(f'Esegui prima un training baseline (es. Training_File_Optimizer_2x2.ipynb)')
    checks.append(('checkpoint', False))
else:
    print(f'\n   [OK] checkpoint: {CKPT_PATH}')
    checks.append(('checkpoint', True))

n_fail = sum(1 for _, ok in checks if not ok)
if n_fail == 0:
    print('\n[OK] ENV CHECK PASSED -- procedi con Cella 2')
else:
    print(f'\n[FAIL] ENV CHECK FAILED -- {n_fail} problemi')
    raise SystemExit('Env check failed')

In [ ]:
# ===========================================================
# CELLA 2 -- Inizializza CFSimulator + lista scenari
# ===========================================================
from utils.simulator import CFSimulator, compute_operational_metrics, aggregate_metrics
from utils.simulator import plot_simulation_static, plot_topdown_snapshot
from utils.simulator import animate_scenario, save_animation
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np

# Setup simulatore: seq_len=200 = 20 secondi (ragionevole per visualizzazione)
# Se vuoi piu' o meno tempo, modifica qui (max 1000 step = 100s).
sim = CFSimulator(
    checkpoint_path=CKPT_PATH,
    cache_path=CACHE_PATH,
    variant='baseline',
    seq_len=200,
)

# Lista scenari disponibili
df_idx = sim.list_scenarios()
print(f'\nTotal scenari: {len(df_idx)}')
print(f'Scenario types: {df_idx.scenario_type.value_counts().to_dict()}')
print(f'Cut-in scenarios: {df_idx.is_cut_in.sum()}')
print(f'\nPrime 10 righe:')
df_idx.head(10)

In [ ]:
# ===========================================================
# CELLA 3 -- Single scenario: static plot + metriche
# ===========================================================
# Cambia SCENARIO_IDX per visualizzare scenari diversi (0..299).
SCENARIO_IDX = 0

result = sim.simulate_scenario(SCENARIO_IDX)
metrics = compute_operational_metrics(result)

# Mostra metriche
print(f'\nScenario idx={SCENARIO_IDX} [{result.scenario_type}]'
      + (' [cut-in]' if result.is_cut_in else ''))
print(f'Params TRUE: ' + ', '.join(f'{k}={v:.2f}' for k, v in result.params_true.items()
                                     if k != 'delta'))
print(f'\n--- Metric operative ---')
for k, v in metrics.items():
    if isinstance(v, (int, float)) and k not in ('idx', 'is_cut_in'):
        if np.isfinite(v):
            print(f'  {k:30s} {v:.4f}')
        else:
            print(f'  {k:30s} inf')

# Plot Layout A statico
fig = plot_simulation_static(result, metrics=metrics)
plt.show()

# Optional: salva PNG
out_png = f'opt_plots/simviz_static_idx{SCENARIO_IDX:03d}.png'
import os
os.makedirs(os.path.dirname(out_png), exist_ok=True)
fig.savefig(out_png, dpi=120, bbox_inches='tight')
print(f'\n[saved] {out_png}')

In [ ]:
# ===========================================================
# CELLA 4 -- Top-down snapshot a frame specifico
# ===========================================================
# Visualizza un singolo istante della simulazione (per intuizione spaziale).
T_FRAME = 100  # frame 100 = 10 secondi (con seq_len=200)
T_FRAME = min(T_FRAME, result.seq_len - 1)

fig_td = plot_topdown_snapshot(result, t_frame=T_FRAME)
plt.show()

out_td = f'opt_plots/simviz_topdown_idx{SCENARIO_IDX:03d}_t{T_FRAME:03d}.png'
fig_td.savefig(out_td, dpi=120, bbox_inches='tight')
print(f'[saved] {out_td}')

In [ ]:
# ===========================================================
# CELLA 5 -- Animazione + export GIF
# ===========================================================
# L'animazione mostra il replay completo dello scenario corrente:
# - LEFT: vista top-down 1D che si aggiorna frame-by-frame
# - RIGHT: 4 sub-panel (gap, velocita', accel, spike raster progressivo)
#           con linea verticale rossa che indica il tempo corrente.
#
# fps=10 = real-time (DT=0.1s -> 10 fps). T frame totale = seq_len.
from IPython.display import HTML, display

anim = animate_scenario(result, fps=10)
# Render inline in notebook (HTML5 video, richiede ffmpeg per video; altrimenti JS)
html_anim = HTML(anim.to_jshtml())  # JS animation, no ffmpeg needed
display(html_anim)
plt.close()  # chiude figura statica (e' gia' embedded nell'animation)

# Export GIF per paper/slide deck
out_gif = f'opt_plots/simviz_anim_idx{SCENARIO_IDX:03d}.gif'
save_animation(anim, out_gif, fps=10)
print(f'\n[GIF salvato] {out_gif}')
print(f'Per .mp4: save_animation(anim, "output.mp4") -- richiede ffmpeg in PATH')

In [ ]:
# ===========================================================
# CELLA 6 -- Multi-scenario aggregate: 20 scenari + tabella riassuntiva
# ===========================================================
# Simula i primi N_SAMPLES scenari della val set, aggrega le metriche
# operative per ottenere statistiche.
N_SAMPLES = 20  # aumenta per piu' statistiche (max 300)

import time
all_metrics = []
t0 = time.time()
for i in range(min(N_SAMPLES, len(df_idx))):
    r = sim.simulate_scenario(i)
    m = compute_operational_metrics(r)
    all_metrics.append(m)
    if (i+1) % 5 == 0:
        print(f'  {i+1}/{N_SAMPLES}  elapsed={time.time()-t0:.1f}s')
print(f'\n[DONE] {len(all_metrics)} simulazioni in {time.time()-t0:.1f}s')

# DataFrame singole scenarios
df_all = pd.DataFrame(all_metrics)
print(f'\nTabella per scenario (head 5):')
display_cols = ['idx', 'scenario_type', 'gap_rmse_m', 'pos_cum_err_m',
                'accel_rmse_masked', 'jerk_max_pred', 'ttc_min_pred_s', 'spike_rate_avg']
df_all[display_cols].head(5)

In [ ]:
# Aggregato + boxplot per categoria
df_summary = aggregate_metrics(all_metrics)
print('TABELLA AGGREGATA:')
key_metric_cols = [c for c in df_summary.columns if any(s in c for s in ['gap_rmse', 'pos_cum_err', 'accel_rmse', 'jerk_max', 'spike_rate_avg']) and c.endswith('_med')]
show_cols = ['group', 'count'] + key_metric_cols
df_summary[show_cols].round(4)

In [ ]:
# Boxplot diagnostici delle 4 metriche chiave
fig, axes = plt.subplots(2, 2, figsize=(13, 8))
metric_plots = [
    ('gap_rmse_m',         'Gap RMSE [m]',          axes[0,0]),
    ('pos_cum_err_m',      'Pos cum. error [m]',    axes[0,1]),
    ('accel_rmse_masked',  'Accel RMSE [m/s²]',     axes[1,0]),
    ('jerk_max_pred',      'Jerk max [m/s³]',       axes[1,1]),
]
for col, ylabel, ax in metric_plots:
    df_all.boxplot(column=col, by='scenario_type', ax=ax)
    ax.set_ylabel(ylabel)
    ax.set_xlabel('scenario_type')
    ax.set_title('')
    ax.grid(alpha=0.3)
fig.suptitle(f'Distribuzione metric operative su {len(df_all)} scenari val', fontsize=12)
plt.tight_layout()
out_box = f'opt_plots/simviz_boxplot_N{N_SAMPLES}.png'
fig.savefig(out_box, dpi=120, bbox_inches='tight')
print(f'[saved] {out_box}')
plt.show()

# Save CSV aggregato per analisi futura
out_csv = f'opt_plots/simviz_metrics_N{N_SAMPLES}.csv'
df_all.to_csv(out_csv, index=False)
print(f'[saved] {out_csv}')

In [ ]:
# ===========================================================
# CELLA 7 -- INTERACTIVE: dropdown scenario + slider tempo (opzionale)
# ===========================================================
# Richiede ipywidgets. Permette di navigare interattivamente tra scenari
# e mostrare snapshot top-down al tempo selezionato.
try:
    from ipywidgets import interact, IntSlider, Dropdown, Output
    HAS_IPW = True
except ImportError:
    print('[WARN] ipywidgets non disponibile -- skip Cella 7')
    HAS_IPW = False

if HAS_IPW:
    # Cache i risultati gia' calcolati (evita ricompute)
    _sim_cache = {m['idx']: None for m in all_metrics}  # placeholder

    def _get_result(idx):
        if _sim_cache.get(idx) is None:
            _sim_cache[idx] = sim.simulate_scenario(idx)
        return _sim_cache[idx]

    out = Output()

    def show_scenario(idx, t_frame):
        with out:
            out.clear_output(wait=True)
            r = _get_result(idx)
            t_frame = min(t_frame, r.seq_len - 1)
            fig = plot_topdown_snapshot(r, t_frame=t_frame, figsize=(13, 4))
            plt.show()
            print(f'idx={idx}  scenario={r.scenario_type}  t={r.time[t_frame]:.1f}s  '
                  f'gap_pred={r.gap_pred[t_frame]:.2f}m  v_pred={r.v_ego_pred[t_frame]:.1f}m/s  '
                  f'a_pred={r.a_pred[t_frame]:+.2f}m/s²  spike%={r.spike_rate[t_frame]*100:.1f}')

    interact(show_scenario,
             idx=Dropdown(options=list(range(min(50, len(df_idx)))), value=0, description='Scenario'),
             t_frame=IntSlider(min=0, max=sim.seq_len-1, value=0, step=1, description='Tempo (tick)'))
    display(out)
    print('\n[ipywidgets attivo] Cambia scenario col dropdown, scorri tempo col slider.')

---
# CUT-IN ANALYSIS -- scenari critici per il deploy

I cut-in sono **gli scenari piu' critici per CF_FSNN** in deploy reale: gap drop brusco (10-30m in 1 tick), il sistema deve reagire entro pochi secondi per evitare collisione. Le celle seguenti caricano una cache `mixed + cut_in_ratio=0.30` (highway/urban/truck mix con 23% cut-in), filtrano gli scenari critici, e producono figure dedicate.

**Cache**: `data/cache_400_mixed_cut0.3_ou0.0.pt` (100 val, 23 cut-in, 4 scenario types).

**Cut-in detection automatica**: heuristic in `engine.py` cerca jump in `s_obs` > 5m in singolo DT. Ritornati `cut_in_t`, `cut_in_gap_before`, `cut_in_gap_after` in SimulationResult. I plot mostrano linea verticale rossa + banda shaded 2s di transient + label CUT-IN.

In [ ]:
# ===========================================================
# CELLA 8 -- Carica cache MIXED + filtra cut-in scenarios
# ===========================================================
CACHE_CUTIN = 'data/cache_400_mixed_cut0.3_ou0.0.pt'
import os
if not os.path.isfile(CACHE_CUTIN):
    print(f'[INFO] {CACHE_CUTIN} non trovato. Genero ora ...')
    from data.generator import generate_dataset
    from config import SEED
    import torch
    train_data = generate_dataset(300, base_seed=SEED, scenario_mix=None,
                                    cut_in_ratio=0.30, noise_scale=0.0)
    val_data = generate_dataset(100, base_seed=SEED+1, scenario_mix=None,
                                  cut_in_ratio=0.30, noise_scale=0.0)
    torch.save({'train': train_data, 'val': val_data, 'seed': SEED}, CACHE_CUTIN)
    print(f'[saved] {CACHE_CUTIN}')

# Re-inizializza simulatore con la cache cut-in (seq_len=700 per catturare
# anche cut-in tardivi -- generator li mette tra 10% e 60% di N=1000)
sim_ci = CFSimulator(
    checkpoint_path=CKPT_PATH,
    cache_path=CACHE_CUTIN,
    variant='baseline',
    seq_len=700,
)
df_ci = sim_ci.list_scenarios()
print()
print(f'Scenari val (mixed cache): {len(df_ci)}')
print(f'  per tipo: {df_ci.scenario_type.value_counts().to_dict()}')
print(f'  cut-in flagged: {df_ci.is_cut_in.sum()}/{len(df_ci)}')

# Filtra cut-in only
cutin_idx = df_ci[df_ci.is_cut_in].index.tolist()
print(f'\nPrimi 10 cut-in indices: {cutin_idx[:10]}')
df_ci[df_ci.is_cut_in].head(10)

In [ ]:
# ===========================================================
# CELLA 9 -- Simula 5 cut-in scenarios + plot statico + metric
# ===========================================================
# Per ogni cut-in produce: 1 static plot (5 panels) con highlight CUT-IN
# + dict metric. Salva PNG in opt_plots/cutin/.
import os
os.makedirs('opt_plots/cutin', exist_ok=True)

N_CUTIN_PLOTS = 5
cutin_results = []
cutin_metrics = []
detected_count = 0
for idx in cutin_idx[:N_CUTIN_PLOTS]:
    r = sim_ci.simulate_scenario(int(idx))
    if r.cut_in_t is None:
        print(f'  idx={idx}: cut-in flag ma evento FUORI dalla window seq_len={sim_ci.seq_len}, SKIP')
        continue
    detected_count += 1
    m = compute_operational_metrics(r)
    cutin_results.append(r); cutin_metrics.append(m)
    fig = plot_simulation_static(r, metrics=m)
    fp = f'opt_plots/cutin/cutin_idx{idx:03d}_{r.scenario_type}.png'
    fig.savefig(fp, dpi=110, bbox_inches='tight')
    plt.close(fig)
    print(f'  [{detected_count}] idx={idx:3d} [{r.scenario_type:7s}] cut-in t={r.time[r.cut_in_t]:.1f}s '
          f'drop {r.cut_in_gap_before:.1f}->{r.cut_in_gap_after:.1f}m  '
          f'gap_rmse={m["gap_rmse_m"]:.2f}m  pos_drift={m["pos_cum_err_m"]:.2f}m  '
          f'TTC_min={m["ttc_min_pred_s"]:.2f}s')
    print(f'      [saved] {fp}')
print(f'\nTotale {detected_count}/{N_CUTIN_PLOTS} cut-in detected e plottati.')

In [ ]:
# ===========================================================
# CELLA 10 -- Animazione cut-in scenario + GIF export
# ===========================================================
if cutin_results:
    r_demo = cutin_results[0]
    print(f'Animating cut-in scenario idx={r_demo.idx} ({r_demo.scenario_type}, '
          f'cut-in at t={r_demo.time[r_demo.cut_in_t]:.1f}s)...')
    anim = animate_scenario(r_demo, fps=10)
    out_gif = f'opt_plots/cutin/cutin_anim_idx{r_demo.idx:03d}.gif'
    save_animation(anim, out_gif, fps=10)
    print(f'[saved] {out_gif}')
    from IPython.display import HTML, display
    display(HTML(anim.to_jshtml()))
    plt.close()
else:
    print('Nessun cut-in detected -- aumenta sim_ci.seq_len o cambia cache.')

In [ ]:
# ===========================================================
# CELLA 11 -- AGGREGATE: tutti i cut-in vs non-cut-in, compare
# ===========================================================
# Simula TUTTI gli scenari (cut-in + non-cut-in) e aggrega le metriche.
# Confronto sistematico per validare se il SNN gestisce bene i cut-in.
import time
all_results = []
all_metrics = []
t0 = time.time()
for i in range(len(df_ci)):
    r = sim_ci.simulate_scenario(i)
    m = compute_operational_metrics(r)
    m['cut_in_detected'] = (r.cut_in_t is not None)
    all_results.append(r)
    all_metrics.append(m)
    if (i+1) % 20 == 0:
        print(f'  {i+1}/{len(df_ci)}  elapsed={time.time()-t0:.1f}s')
elapsed = time.time() - t0
print(f'\n[DONE] {len(all_metrics)} simulazioni in {elapsed:.1f}s ({elapsed/len(all_metrics):.2f}s/sim)')

import pandas as pd
df_all = pd.DataFrame(all_metrics)

print('\n=== Confronto metric: cut-in DETECTED vs no-cut-in ===')
summary_cols = ['gap_rmse_m', 'gap_max_err_m', 'pos_cum_err_m',
                'accel_rmse_masked', 'jerk_max_pred', 'jerk_p95_pred',
                'ttc_min_pred_s', 'spike_rate_avg']
for col in summary_cols:
    if col not in df_all.columns: continue
    s_ci = df_all[df_all.cut_in_detected][col]
    s_no = df_all[~df_all.cut_in_detected][col]
    if len(s_ci) == 0 or len(s_no) == 0: continue
    delta_pct = (s_ci.median() / s_no.median() - 1) * 100 if s_no.median() != 0 else float('inf')
    print(f'  {col:25s}  cut-in: med={s_ci.median():.4f}  no: med={s_no.median():.4f}  '
          f'd={delta_pct:+5.1f}%')

fig, axes = plt.subplots(2, 2, figsize=(13, 8))
metric_plots = [
    ('gap_rmse_m',         'Gap RMSE [m]',          axes[0,0]),
    ('pos_cum_err_m',      'Pos cum. error [m]',    axes[0,1]),
    ('jerk_max_pred',      'Jerk max [m/s3]',       axes[1,0]),
    ('accel_rmse_masked',  'Accel RMSE [m/s2]',     axes[1,1]),
]
for col, ylabel, ax in metric_plots:
    if col not in df_all.columns: continue
    data = [df_all[~df_all.cut_in_detected][col].values,
            df_all[df_all.cut_in_detected][col].values]
    ax.boxplot(data, labels=['no-cut-in', 'cut-in'], showfliers=False)
    ax.set_ylabel(ylabel)
    ax.grid(alpha=0.3)
fig.suptitle(f'Confronto metric: cut-in DETECTED (n={df_all.cut_in_detected.sum()}) vs '
             f'no-cut-in (n={(~df_all.cut_in_detected).sum()}) -- baseline ALIF+BPTT',
             fontsize=11)
plt.tight_layout()
out_box = 'opt_plots/cutin/boxplot_cutin_vs_normal.png'
fig.savefig(out_box, dpi=120, bbox_inches='tight')
plt.show()
print(f'\n[saved] {out_box}')

print('\n=== VERDETTO OPERATIVO ===')
ci_gap_med = df_all[df_all.cut_in_detected].gap_rmse_m.median()
no_gap_med = df_all[~df_all.cut_in_detected].gap_rmse_m.median()
ratio = ci_gap_med / no_gap_med if no_gap_med != 0 else float('inf')
if ratio < 1.5:
    print(f'  OK Cut-in gestiti acceptably (gap_rmse cut-in {ci_gap_med:.3f} vs normal {no_gap_med:.3f}, ratio {ratio:.1f}x)')
elif ratio < 3.0:
    print(f'  WARN Cut-in degradano performance (gap_rmse cut-in {ci_gap_med:.3f} vs normal {no_gap_med:.3f}, ratio {ratio:.1f}x)')
else:
    print(f'  FAIL Cut-in problematici (gap_rmse cut-in {ci_gap_med:.3f} vs normal {no_gap_med:.3f}, ratio {ratio:.1f}x)')
ci_pos_med = df_all[df_all.cut_in_detected].pos_cum_err_m.median()
print(f'  Cumulative position error cut-in median: {ci_pos_med:.2f} m')
ttc_finite = df_all[df_all.cut_in_detected].ttc_min_pred_s
ttc_finite = ttc_finite[ttc_finite < 1e9]
if len(ttc_finite) > 0:
    print(f'  TTC min cut-in (escluso inf): median={ttc_finite.median():.2f}s p10={ttc_finite.quantile(0.1):.2f}s')
df_all.to_csv('opt_plots/cutin/aggregate_metrics.csv', index=False)
print(f'  [saved] opt_plots/cutin/aggregate_metrics.csv')